# Explore Vector Database recipes.lance

## 1. Retrieval from Vector Database

In this step, we test semantic search using the vector database (LanceDB).

The goal is to retrieve the most relevant recipes given a user query.

In [15]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

import lancedb
from sentence_transformers import SentenceTransformer
from src.cookmate.utils.config import DB_DIR

## 2. Load embedding model

We use the same embedding model used during ingestion to ensure consistency.

In [16]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5857.89it/s]


## 3. Connect to LanceDB

We connect to the local vector database and open the recipes table.

In [17]:
db = lancedb.connect(DB_DIR)
table = db.open_table("recipes")

## 4. Define user query

We simulate a user query to test semantic retrieval.

In [18]:
query = "chicken pasta with tomato"

## 5. Convert query to embedding

We transform the query into a vector representation.

In [19]:
query_vector = model.encode(query)

## 6. Perform similarity search

We retrieve the most similar recipes from the database.

In [20]:
result = table.search(query_vector).limit(3).to_pandas()
result

,id,text,vector,_distance
0,recipe_1735,Recipe: Chicken Spaghetti. Ingredients: stalks...,"[-0.058917366, -0.058045264, 0.006024487, -0.0...",0.516375
1,recipe_1635,"Recipe: Chicken, Pesto And Sun-Dried Tomato Pa...","[-0.08670863, -0.05756682, 0.027218172, -0.009...",0.572196
2,recipe_1065,Recipe: Pasta In Tomato Cream Sauce. Ingredien...,"[-0.068504214, -0.032079052, 0.026498469, 0.00...",0.642061


## 7. Results

The retrieved recipes are semantically similar to the query.
This demonstrates that the system can find relevant recipes even without exact keyword matching.
The distance metric is also quite good.

In this experiment, we use local embeddings for ingestion to avoid rate limits.
We tested Cohere embeddings, but they could not perform well due to API limitations and rate limits.

For this reason, we rely on local embeddings for retrieval.

Considering the relatively small size of the dataset, the current approach already provides good retrieval results.

## 8. Retrieval-Augmented Generation (RAG)

In this step, we combine retrieval with a language model.

The system retrieves relevant recipes and uses an LLM to generate a final response.

In [21]:
import cohere 
import os

co = cohere.Client(os.getenv("COHERE_API_KEY"))

## 9. Prepare context from retrieved recipes

We take the retrieved recipes and build a context for the LLM.

In [22]:
top_recipe = result

context = ""

for _, row in top_recipe.iterrows():
    context += f"{row['text']}\n\n"

print(context[:500])

Recipe: Chicken Spaghetti. Ingredients: stalks celery, yellow onion, tomatoes, cream of mushroom soup, chicken breasts, Velveeta cheese, extra virgin olive oil, spaghettini. Instructions: First, place the chicken breasts in a pot with enough water to cover them. Boil the chicken until completely cooked. Remove from heat, pour chicken stock into a bowl and set aside for later. Set chicken breasts aside to cool and chop the vegetables. By the time you're finished chopping, the chicken should be co


## 10. Generate response with LLM

Now we pass the user query and retrieved context to the model.
For the answer to the user we use Openrouter models.

In [24]:
from dotenv import load_dotenv
from pydantic_ai import Agent

load_dotenv()

recipe_agent = Agent(
    model="openrouter:liquid/lfm-2.5-1.2b-instruct:free",
    system_prompt="""
    You are a helpful cooking assistant.

    Use only the provided recipe context to answer.
    Suggest recipes based on the user's ingredients.
    Give clear step-by-step instructions.
    If the context does not contain enough information, say so.
    """,
    retries=1,
)

## Rag 

We test a request from the user and we build the answer with LLm

In [26]:
query = "I have a chicken and a pasta at home suggest a recipe"

prompt = f"""
    Usre request: {query}
    Retrived recipe context: {context}
"""

result = await recipe_agent.run(prompt)

print(result.output)

Based on the ingredients you have (chicken and pasta), here are a couple of recipe ideas you can try:

### 1. Chicken with Tomato Sauce Pasta
From your chicken recipe context, you could easily make this as a quick chicken spaghetti:

**Ingredients:**
- Chicken breasts  
- Spaghetti or penne  
- Fresh tomatoes  
- Basil  
- Cream or light cream  
- Olive oil  
- Salt and pepper  
- Parmesan cheese (for serving)

**Instructions:**
1. Cook spaghetti according to package directions, then keep it warm.
2. While pasta cooks, heat olive oil in a large pan over medium heat.
3. Add the cooked chicken, cook for 5–7 minutes per side until done.
4. Pour in sliced tomatoes and let them simmer for 5–10 minutes until softened.
5. Stir in fresh basil and the cream (or half and half if you prefer a lighter sauce).
6. Add the cooked spaghetti to the pan and mix well.
7. Season with salt and pepper, then sprinkle with Parmesan cheese.
8. Serve immediately and enjoy!

### 2. Chicken Pesto with Sun-Dried T